# RaceShift FFR Colab Training
Deep multi-layer Forward-Forward regression for next-lap forecasting. No global backpropagation is used in the RaceShift FFR training path.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Upload/unzip the RaceShift project
Upload the ZIP to Colab, unzip it under `/content/RaceShift`, then run the next cells.


In [ ]:
%cd /content/RaceShift
!pip install -e '.[research]'


## Collect a smoke-test subset first
Start with a few races before downloading full seasons.


In [ ]:
!python scripts/fetch_fastf1_seasons.py --years 2022-2025 --session R --max-events 3 --output /content/drive/MyDrive/RaceShiftData/raw --cache /content/drive/MyDrive/RaceShiftData/cache


In [ ]:
import pandas as pd, glob
files=glob.glob('/content/drive/MyDrive/RaceShiftData/raw/*.parquet')
raw=pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
raw.to_parquet('/content/drive/MyDrive/RaceShiftData/f1_laps.parquet', index=False)
len(raw), raw.head()


## Train the production 4-layer FFR model
Architecture: 512 → 384 → 256 → 192 hidden nodes.


In [ ]:
!python scripts/train_ffr.py --input /content/drive/MyDrive/RaceShiftData/f1_laps.parquet --config configs/ffr_production.json --output /content/drive/MyDrive/RaceShiftData/artifacts/raceshift_ffr --train-end 2023 --val-year 2024 --test-year 2025


## Optional large-width ablation
Only run after the production configuration is validated.


In [ ]:
#!python scripts/train_ffr.py --input /content/drive/MyDrive/RaceShiftData/f1_laps.parquet --config configs/ffr_colab_large.json --output /content/drive/MyDrive/RaceShiftData/artifacts/raceshift_ffr_large --train-end 2023 --val-year 2024 --test-year 2025
